# 0.6.1 - XGBoost Location Evaluation - Saint-Imier

Evaluates the 0.5 model on the test observations of one municipality, at the thresholds matched to the radon map legend boundaries in 0.5.1 (confusion matrix at the decision threshold).

## Setup

In [ ]:
import json
import pickle

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score

from evaluation import (
    bootstrap_ci,
    load_test_egid,
    load_test_set,
    operating_points,
    plot_confusion_matrix,
    plot_precision_recall,
    regression_scores,
    test_scores,
)

ROOT = '../../'
DIR = '../../models/'
LOCATION = 'Saint-Imier'

## Data and model

In [ ]:
X_test, y_test = load_test_set(ROOT)
with open(f'{DIR}0.2-encoding-pipeline.pkl', 'rb') as file:
    preprocessor = pickle.load(file)

model = xgb.XGBRegressor()
model.load_model(f'{DIR}0.5-xgboost-log-model.json')
y_pred = pd.Series(np.expm1(model.predict(preprocessor.transform(X_test))), index=y_test.index)

# Settings chosen in 0.5.1
with open(f'{DIR}0.5-decision-threshold.json') as file:
    settings = json.load(file)
RADON_REFERENCE_LEVEL, DECISION_THRESHOLD = settings['reference_level'], settings['decision_threshold']
# Thresholds matched to the recall of the radon map at its legend boundaries; the decision threshold is the > 1 % one
THRESHOLDS = {f'Map {cutoff}': threshold for cutoff, threshold in settings['map_matched_thresholds'].items()}

In [ ]:
X_test['Rdn - Commune'].value_counts().head(10)

## Evaluation

In [ ]:
def subset_summary(name, mask):
    exceeds = y_test[mask] >= RADON_REFERENCE_LEVEL
    return {
        'Subset': name,
        'Observations': int(mask.sum()),
        'Prevalence (%)': round(exceeds.mean() * 100, 1),
        **regression_scores(y_test[mask], y_pred[mask]),
        'Average precision': round(average_precision_score(exceeds, y_pred[mask]), 3),
    }


def subset_operating_points(name, mask):
    exceeds = y_test[mask] >= RADON_REFERENCE_LEVEL
    points = operating_points(exceeds, y_pred[mask], THRESHOLDS)
    points.insert(0, 'Subset', name)
    return points


def plot_subset(name, mask, path=None):
    """Precision-recall curve and confusion matrix at the decision threshold; saved to `path` + '-precision-recall' / '-confusion-matrix'."""
    exceeds = y_test[mask] >= RADON_REFERENCE_LEVEL
    plot_precision_recall(exceeds, y_pred[mask], f'Precision–recall, test subset - {name}', path=path and f'{path}-precision-recall')
    plot_confusion_matrix(exceeds, y_pred[mask] >= DECISION_THRESHOLD, f'{name}, decision threshold {DECISION_THRESHOLD:.0f} Bq/m3', RADON_REFERENCE_LEVEL,
                          path=path and f'{path}-confusion-matrix')

In [ ]:
mask = X_test['Rdn - Commune'] == LOCATION
subsets = {'All test observations': pd.Series(True, index=X_test.index), LOCATION: mask}
pd.DataFrame([subset_summary(name, m) for name, m in subsets.items()])

In [ ]:
pd.concat([subset_operating_points(name, m) for name, m in subsets.items()], ignore_index=True)

### Confidence intervals

Bootstrap over the test buildings of the municipality (1,000 samples), at the decision threshold.

In [ ]:
egid_test = load_test_egid(ROOT)
print(f'{egid_test[mask].nunique()} test buildings')
bootstrap_ci(lambda t, p: test_scores(t, p, DECISION_THRESHOLD, RADON_REFERENCE_LEVEL), y_test[mask], y_pred[mask], egid_test[mask])

In [ ]:
plot_subset(LOCATION, mask, path=f'{ROOT}reports/figure-st-imier')